### Leitura do arquivo CSV com módulo nativo

In [617]:
import csv
import json
from datetime import datetime, date

SUSPICIOUS_LIMIT = 10000.00

### Organização do código em funções


In [618]:
def read_transactions(file_name):
    """Lê o CSV e retorna as linhas brutas."""
    try:
        with open(file_name, "r", encoding="utf-8") as arquivo:
            return list(csv.DictReader(arquivo))
    except FileNotFoundError:
        print(f"Erro: arquivo '{file_name}' não encontrado.")
        return []

In [619]:
def validate_transaction(row):
    """Valida uma transação e retorna o registro limpo."""
    try:
        try:
            date = datetime.strptime(row["data"], "%Y-%m-%d")
        except ValueError:
            return None

        if not row["id"] or not row["id"].isdigit():
            return None

        if not row["cliente_id"].strip():
            return None

        type = row["tipo"].lower()
        if type not in ("credito", "debito"):
            return None

        try:
            value = float(row["valor"])
            if value <= 0:
                return None
        except ValueError:
            return None

        return {
            "id": int(row["id"]),
            "cliente_id": row["cliente_id"].strip(),
            "data": date,
            "tipo": type,
            "valor": value
        }

    except (ValueError, KeyError):
        return None


In [620]:
def calculate_metrics(transactions):
    """Calcula as métricas mensais e identifica transações suspeitas."""
    monthly_report = { }
    suspects_transactions = []
    dates = [t["data"] for t in transactions]

    oldest_date = min(dates)
    newest_date = max(dates)

    days_between = (newest_date - oldest_date).days

    for transaction in transactions:

        month = transaction["data"].strftime("%Y-%m")

        if month not in monthly_report:
            monthly_report[month] = {
                "quantidade": 0,
                "total_credito": 0.0,
                "total_debito": 0.0,
                "saldo": 0.0,
                "maior_valor": transaction["valor"],
                "menor_valor": transaction["valor"]
            }

        monthly_report[month]["quantidade"] += 1

        if transaction["tipo"] == "credito":
            monthly_report[month]["total_credito"] += transaction["valor"]
        else:
            monthly_report[month]["total_debito"] += transaction["valor"]

        monthly_report[month]["maior_valor"] = max(
            monthly_report[month]["maior_valor"],
            transaction["valor"]
            )

        monthly_report[month]["menor_valor"] = min(
            monthly_report[month]["menor_valor"],
            transaction["valor"]
            )

        if (transaction["valor"] > SUSPICIOUS_LIMIT):
            suspects_transactions.append(transaction)
    
    
    for month, data in monthly_report.items():
        balance = data["total_credito"] - data["total_debito"]

        data["saldo"] = balance
        data["media"] = round((
            data["total_credito"] + data["total_debito"]
        ) / data["quantidade"], 2)

    return {
        "mensal": monthly_report,
        "transacoes_suspeitas": suspects_transactions,
        "data_mais_antiga": oldest_date,
        "data_mais_recente": newest_date,
        "dias_entre": days_between
    }

In [621]:
def save_json(report, filename="report.json"):
    """Salva o relatório em JSON."""
    with open(filename, "w", encoding="utf-8") as arquivo:
        json.dump(report, arquivo, indent=4, ensure_ascii=False,default=str)

In [622]:
def display_report(report):
    """Exibe o relatório no console."""
    print("===== RELATÓRIO MENSAL =====\n")

    for month, data in sorted(report["mensal"].items()):
        print(f"Mês: {month}")
        print(f"  Transações: {data['quantidade']}")
        print(f"  Total crédito: R$ {data['total_credito']:,.2f}")
        print(f"  Total débito:  R$ {data['total_debito']:,.2f}")
        print(f"  Saldo:         R$ {data['saldo']:,.2f}")
        print(f"  Média:         R$ {data['media']:,.2f}")
        print(f"  Maior valor:   R$ {data['maior_valor']:,.2f}")
        print(f"  Menor valor:   R$ {data['menor_valor']:,.2f}")
        print()

    if report["transacoes_suspeitas"]:
        print('===== TRANSAÇÕES SUSPEITAS =====')
        for transaction in report["transacoes_suspeitas"]:
            print(f"ID: {transaction['id']} | Cliente: {transaction['cliente_id']} | Data: {transaction['data']} | Valor: R$ {transaction['valor']:,.2f}")
        print()
    else:
        print("Nenhuma transação suspeita encontrada.")
        print()
        
    print("===== PERÍODO ANALISADO =====")
    print(
        f"Data mais antiga: "
        f"{report['data_mais_antiga'].strftime('%Y-%m-%d')}"
    )
    print(
        f"Data mais recente: "
        f"{report['data_mais_recente']}"
    )
    print(f"Dias entre elas: {report['dias_entre']}")

In [623]:
file_url = "../../work/documents/transactions.csv"

df = read_transactions(file_url)

### Validação e limpeza dos dados

In [624]:
total_rows = 0
valid_rows = 0
invalid_rows = 0
valid_transactions = []

for row in df:
    total_rows += 1
    cleaned_row = validate_transaction(row)
    if cleaned_row:
        valid_transactions.append(cleaned_row)
        
        valid_rows += 1
    else:
        invalid_rows += 1

In [625]:
print(f"Total de linhas lidas: {total_rows}")
print(f"Linhas válidas: {valid_rows}")
print(f"Linhas inválidas: {invalid_rows}")

Total de linhas lidas: 20
Linhas válidas: 15
Linhas inválidas: 5


### Agrupamento mensal e métricas


In [626]:
report_by_month = calculate_metrics(valid_transactions)

### Exportação do relatório em JSON


In [627]:
relatorio = {
    "gerado_em": date.today().isoformat(),
    "total_transacoes_validas": valid_rows,
    "total_transacoes_invalidas": invalid_rows,
    "resumo_mensal": report_by_month['mensal'],
    "transacoes_suspeitas": [
        {
            "id": t["id"],
            "cliente_id": t["cliente_id"],
            "data": t["data"].strftime("%Y-%m-%d"),
            "valor": t["valor"]
        }
        for t in report_by_month['transacoes_suspeitas']
    ]
}

save_json(relatorio,"relatorio.json")
display_report(report_by_month)

===== RELATÓRIO MENSAL =====

Mês: 2026-01
  Transações: 3
  Total crédito: R$ 3,500.00
  Total débito:  R$ 430.50
  Saldo:         R$ 3,069.50
  Média:         R$ 1,310.17
  Maior valor:   R$ 3,500.00
  Menor valor:   R$ 180.50

Mês: 2026-02
  Transações: 3
  Total crédito: R$ 15,000.00
  Total débito:  R$ 740.90
  Saldo:         R$ 14,259.10
  Média:         R$ 5,246.97
  Maior valor:   R$ 15,000.00
  Menor valor:   R$ 320.00

Mês: 2026-03
  Transações: 4
  Total crédito: R$ 15,500.00
  Total débito:  R$ 879.90
  Saldo:         R$ 14,620.10
  Média:         R$ 4,094.97
  Maior valor:   R$ 12,000.00
  Menor valor:   R$ 99.90

Mês: 2026-04
  Transações: 3
  Total crédito: R$ 4,300.00
  Total débito:  R$ 770.45
  Saldo:         R$ 3,529.55
  Média:         R$ 1,690.15
  Maior valor:   R$ 4,300.00
  Menor valor:   R$ 210.45

Mês: 2026-05
  Transações: 2
  Total crédito: R$ 2,750.00
  Total débito:  R$ 89.99
  Saldo:         R$ 2,660.01
  Média:         R$ 1,419.99
  Maior valor:   R$ 2,7